# Lumis 05: Grad-CAM check

Grad-CAM shows which parts of an image drove the model's prediction (red = most influence). This checks that EfficientNet-B0 looks at the skin, not at the **©Dermnet.com** watermark printed across the middle of every DermNet image. If it relied on the watermark, its accuracy would not carry over to real user photos.

Two grids, both on validation images (the test set stays untouched):
1. **Random sample:** 5 images per class
2. **Confident mistakes:** the 10 wrong predictions the model was most sure about. Shortcuts usually show up here first

**What to look for:** heat on the rash, lesion or affected skin is good. Heat that consistently traces the watermark text, or sits on the image edges and background, is a red flag.

**T4 GPU** recommended.

## 1. Setup

In [ ]:
RUN = None   # None = the latest EfficientNet-B0 run
N_PER_CLASS = 5
SEED = 42

from google.colab import drive
drive.mount('/content/drive')

import os, json, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

BASE = '/content/drive/MyDrive/Lumis'
LOCAL = '/content/dermnet'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if RUN is None:
    RUN = runs[runs.backbone == 'efficientnetb0'].iloc[-1]['run']
info = runs[runs.run == RUN].iloc[-1]
RUN_DIR = f'{BASE}/exports/runs/{RUN}'
cal = json.load(open(f'{RUN_DIR}/calibration.json'))
T, CLASSES = cal['temperature'], cal['classes']
size = int(info.img_size)

model = keras.models.load_model(info.model_file)
base = [l for l in model.layers if isinstance(l, keras.Model)][0]
gap = [l for l in model.layers if isinstance(l, keras.layers.GlobalAveragePooling2D)][0]
dense = model.get_layer('logits')
PREPROCESS = {'mobilenetv2': keras.applications.mobilenet_v2.preprocess_input,
              'efficientnetb0': keras.applications.efficientnet.preprocess_input,
              'resnet50': keras.applications.resnet50.preprocess_input}[info.backbone]

split = pd.read_csv(f"{BASE}/data/lumis_split_{info.get('split', 'v1') if isinstance(info.get('split', 'v1'), str) else 'v1'}.csv")
val_df = split[split.split == 'val'].reset_index(drop=True)
print('Loaded', RUN, '| temperature', T)

## 2. Grad-CAM

In [ ]:
def load_image(rel_path):
    img = tf.io.decode_image(tf.io.read_file(os.path.join(LOCAL, rel_path)), channels=3, expand_animations=False)
    img = tf.cast(tf.image.resize(img, (size, size)), tf.uint8)
    return img.numpy(), PREPROCESS(tf.cast(img, tf.float32))[None]

def gradcam(x):
    with tf.GradientTape() as tape:
        conv = base(x, training=False)          # last convolutional feature map
        tape.watch(conv)
        logits = dense(gap(conv))
        cls = int(tf.argmax(logits[0]))
        score = logits[:, cls]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(conv * weights, axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (size, size))[..., 0].numpy()
    probs = tf.nn.softmax(logits[0] / T).numpy()
    return cam, cls, probs

def show_grid(rows, title, path, cols=5):
    n = len(rows)
    fig, axes = plt.subplots(int(np.ceil(n / cols)), cols, figsize=(3.2 * cols, 3.6 * np.ceil(n / cols)), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (_, r) in zip(axes.flat, rows.iterrows()):
        raw, x = load_image(r.path)
        cam, cls, probs = gradcam(x)
        ax.imshow(raw)
        ax.imshow(cam, cmap='jet', alpha=0.4)
        ok = CLASSES[cls] == r.lumis_class
        ax.set_title(f'true: {r.lumis_class}\npred: {CLASSES[cls]} ({probs[cls]:.0%})',
                     fontsize=9, color='black' if ok else 'red')
    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(path, dpi=80)
    plt.show()

## 3. Random sample: 5 validation images per class

In [ ]:
sample = (val_df.groupby('lumis_class', group_keys=False)
          .apply(lambda g: g.sample(N_PER_CLASS, random_state=SEED)))
show_grid(sample, 'Grad-CAM: random validation images (red titles = wrong prediction)',
          f'{RUN_DIR}/gradcam_random.jpg')

## 4. Confident mistakes

Uses the validation predictions saved by notebook 03, calibrated with the temperature from notebook 04.

In [ ]:
logits = np.load(f'{RUN_DIR}/val_logits.npy')
y = np.load(f'{RUN_DIR}/val_labels.npy')
z = logits / T
probs = np.exp(z - z.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
pred, conf = probs.argmax(1), probs.max(1)
wrong = np.where(pred != y)[0]
top = wrong[np.argsort(-conf[wrong])][:10]
print('Most confident mistakes: calibrated confidence', np.round(conf[top], 2))
show_grid(val_df.iloc[top], 'Grad-CAM: the 10 most confident mistakes',
          f'{RUN_DIR}/gradcam_confident_mistakes.jpg')
print('Saved both grids to', RUN_DIR)